In [ ]:
from dotenv import load_dotenv
import glob
import os

from langchain_teddynote import logging
from langchain_teddynote.korean import stopwords
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_upstage import UpstageEmbeddings

from pinecone import PodSpec
from langchain_teddynote.community.pinecone import preprocess_documents, create_index
from langchain_teddynote.community.pinecone import create_sparse_encoder, fit_sparse_encoder, load_sparse_encoder
from langchain_teddynote.community.pinecone import upsert_documents, upsert_documents_parallel
from langchain_teddynote.community.pinecone import delete_namespace, delete_by_filter
from langchain_teddynote.community.pinecone import init_pinecone_index
from langchain_teddynote.community.pinecone import PineconeKiwiHybridRetriever

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-09")

In [ ]:
# 한글 불용어 사전 불러오기 (불용어 사전 출처: https://www.ranks.nl/stopwords/korean)
stopword = stopwords()
stopword[:20]

데이터 전처리

In [ ]:
# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

In [ ]:
split_docs = []

# 텍스트 파일을 load -> List[Document] 형태로 변환
files = sorted(glob.glob("data/*.pdf"))

for file in files:
    loader = PyMuPDFLoader(file)
    split_docs.extend(loader.load_and_split(text_splitter))

# 문서 개수 확인
len(split_docs)

In [ ]:
split_docs[0].page_content

In [ ]:
split_docs[0].metadata

문서 전처리

In [ ]:
contents, metadatas = preprocess_documents(
    split_docs=split_docs,
    metadata_keys=["source", "page", "author"],
    min_length=5,
    use_basename=True
)

In [ ]:
# use_basename=True 일 때, source 키에 파일명만 저장됩니다.(디렉토리를 제외됩니다.)
metadatas["source"][:5]

In [ ]:
# VectorStore 에 저장할 문서 확인
contents[:5]

In [ ]:
# VectorStore 에 저장할 metadata 확인
metadatas.keys()

In [ ]:
# metadata 에서 source 를 확인합니다.
metadatas["source"][:5]

In [ ]:
# 문서 개ㄴㅁ수 확인, 소스 개수 확인, 페이지 개수 확인
len(contents), len(metadatas["source"]), len(metadatas["page"])

VectorStore 인덱스 생성

In [ ]:
pc_index = create_index(
    api_key=os.environ["PINECONE_API_KEY"],
    index_name="teddynote-db-index",  # 인덱스 이름을 지정합니다.
    dimension=4096,  # Embedding 차원과 맞춥니다. (OpenAIEmbeddings: 1536, UpstageEmbeddings: 4096)
    metric="dotproduct",  # 유사도 측정 방법을 지정합니다. (dotproduct, euclidean, cosine)
)

Sparse encoder 생성

In [ ]:
sparse_encoder = create_sparse_encoder(stopwords(), mode="kiwi")  # # 한글 불용어 사전 + Kiwi 형태소 분석기를 사용합니다.

In [ ]:
# Sparse encoder로 contents 학습
saved_path = fit_sparse_encoder(sparse_encoder=sparse_encoder, contents=contents, save_path="./sparse_encoder.pkl")

In [ ]:
# 학습된 sparse encoder 불러오기
sparse_encoder = load_sparse_encoder("./sparse_encoder.pkl")

Upsert: DB index에 추가

In [ ]:
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
upstage_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")

In [ ]:
%%time
upsert_documents(
    index=pc_index,  # Pinecone 인덱스
    namespace="teddynote-namespace-01",  # Pinecone namespace
    contents=contents,  # 이전에 전처리한 문서 내용
    metadatas=metadatas,  # 이전에 전처리한 문서 메타데이터
    sparse_encoder=sparse_encoder,  # Sparse encoder
    embedder=upstage_embeddings,
    batch_size=32
)

In [ ]:
%%time
upsert_documents_parallel(
    index=pc_index,  # Pinecone 인덱스
    namespace="teddynote-namespace-02",  # Pinecone namespace
    contents=contents,  # 이전에 전처리한 문서 내용
    metadatas=metadatas,  # 이전에 전처리한 문서 메타데이터
    sparse_encoder=sparse_encoder,  # Sparse encoder
    embedder=upstage_embeddings,
    batch_size=64,
    max_workers=30
)

인덱스 조회 및 삭제

In [ ]:
# 인덱스 조회
pc_index.describe_index_stats()

In [ ]:
# namespace 삭제
delete_namespace(pinecone_index=pc_index, namespace="teddynote-namespace-01")

In [ ]:
pc_index.describe_index_stats()

In [ ]:
# metadata 필터링 으로 삭제하기 (유료 사용자 전용 기능)
delete_by_filter(
    pinecone_index=pc_index,
    namespace="teddynote-namespace-02",
    filter={"source": {"$eq": "SPRi AI Brief_8월호_산업동향.pdf"}}
)

In [ ]:
pc_index.describe_index_stats()

Retriever

In [ ]:
pinecone_params = init_pinecone_index(  # Pinecone 인덱스를 초기화하고 필요한 구성 요소를 설정
    index_name="teddynote-db-index",  # Pinecone 인덱스 이름
    namespace="teddynote-namespace-02",  # Pinecone Namespace
    api_key=os.environ["PINECONE_API_KEY"],  # Pinecone API Key
    sparse_encoder_path="./sparse_encoder.pkl",  # Sparse Encoder 저장경로(save_path)
    stopwords=stopwords(),  # 불용어 사전
    tokenizer="kiwi",
    embeddings=UpstageEmbeddings(model="solar-embedding-1-large-query"),  # Dense Embedder
    top_k=5,  # Top-K 문서 반환 개수
    alpha=0.5,  # alpha=0.75로 설정한 경우, (0.75: Dense Embedding, 0.25: Sparse Embedding)
)

In [ ]:
# Pinecone과 Kiwi를 결합한 하이브리드 retriever
pinecone_retriever = PineconeKiwiHybridRetriever(**pinecone_params)

In [ ]:
# 일반 검색
search_results = pinecone_retriever.invoke("gpt-4o 미니 출시 관련 정보에 대해서 알려줘")
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")

In [ ]:
# 최대 반환할 문서 수 지정
search_results_k = pinecone_retriever.invoke(
    "gpt-4o 미니 출시 관련 정보에 대해서 알려줘", 
    search_kwargs={"k": 1}
)
for result_k in search_results_k:
    print(result_k.page_content)
    print(result_k.metadata)
    print("\n====================\n")

In [ ]:
# 벡터 가중치 조절 (밀집 벡터와 희소 벡터의 가중치. 0.5가 기본값. 1에 가까울수록 dense 벡터의 가중치가 높아짐.)
search_results_a1 = pinecone_retriever.invoke(
    "앤스로픽", 
    search_kwargs={"alpha": 1, "k": 1}
)
for result_a1 in search_results_a1:
    print(result_a1.page_content)
    print(result_a1.metadata)
    print("\n====================\n")

In [ ]:
search_results_a0 = pinecone_retriever.invoke(
    "앤스로픽", 
    search_kwargs={"alpha": 0, "k": 1}
)
for result_a0 in search_results_a0:
    print(result_a0.page_content)
    print(result_a0.metadata)
    print("\n====================\n")

In [ ]:
# page가 5보다 작은 문서만 검색
search_results_meta1 = pinecone_retriever.invoke(
    "앤스로픽의 claude 출시 관련 내용을 알려줘", 
    search_kwargs={
        "filter": {"page": {"$lt": 5}},  # less than
        "k": 2
    }
)
for result_meta1 in search_results_meta1:
    print(result_meta1.page_content)
    print(result_meta1.metadata)
    print("\n====================\n")

In [ ]:
# SPRi AI Brief_8월호_산업동향.pdf 문서 내에서 검색
search_results_meta2 = pinecone_retriever.invoke(
    "앤스로픽의 claude 출시 관련 내용을 알려줘", 
    search_kwargs={
        "filter": {"source": {"$eq": "SPRi AI Brief_8월호_산업동향.pdf"}},  # equal
        "k": 3
    }
)
for result_meta2 in search_results_meta2:
    print(result_meta2.page_content)
    print(result_meta2.metadata)
    print("\n====================\n")

Reranking 적용 (의존성 해결 필요)

In [ ]:
# reranker 미사용
retrieval_results = pinecone_retriever.invoke(
    "앤스로픽의 클로드 소넷",
)

# BGE-reranker-v2-m3 모델 사용
reranked_results = pinecone_retriever.invoke(
    "앤스로픽의 클로드 소넷", 
    search_kwargs={
        "rerank": False, 
        "rerank_model": "bge-reranker-v2-m3", 
        "top_n": 3
    }
)

In [ ]:
# Reranker 적용 안한것과 적용한것 비교
for res1, res2 in zip(retrieval_results, reranked_results):
    print("[Retrieval]")
    print(res1.page_content)
    print("\n------------------\n")
    print("[Reranked] rerank_score: ", res2.metadata["rerank_score"])
    print(res2.page_content)
    print("\n====================\n")